# **Лабораторная работа №5. Табличные данные: корректный импорт CSV/Excel**

---

Складские остатки смешанные типы в quantity, дубликаты, лишние пробелы sku, date, quantity, warehouse



## **Создание Colab‑ноутбука и импорт библиотек**

In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_colwidth', None)

## **Подготовка демонстрационных файлов CSV и Excel (внутри Colab)**

In [3]:
rows = [
    {'sku': 1000, 'date': '2025.01.01', 'quantity': 10, 'warehouse': 'Almaty'},
    {'sku': 1050, 'date': '2025.01.02', 'quantity': 25, 'warehouse': 'Kostanay'},
    {'sku': 1809, 'date': '2025.01.03', 'quantity': 19, 'warehouse': 'Kokshetau '},
    {'sku': 1309, 'date': '2025.01.04', 'quantity': 29, 'warehouse': 'Astana'},
    {'sku': 2348, 'date': '2025.01.05', 'quantity': '10', 'warehouse': 'Almaty'},
    {'sku': 2348, 'date': '2025.01.05', 'quantity': 5, 'warehouse': 'Kostanay '},
    {'sku': 8437, 'date': '2025.01.07', 'quantity': 90, 'warehouse': 'Almaty'},
    {'sku': 3478, 'date': '2025.01.08', 'quantity': '10', 'warehouse': 'Astana '},
    {'sku': 2498, 'date': '2025.01.09', 'quantity': 129, 'warehouse': 'Almaty'},
    {'sku': 2348, 'date': '2025.01.10', 'quantity': 48, 'warehouse': 'Kokshetau'},
    {'sku': 5849, 'date': '2025.01.11', 'quantity': 1, 'warehouse': 'Kostanay'},
    {'sku': 5849, 'date': '2025.01.11', 'quantity': '29', 'warehouse': 'Kyzylorda'},
    {'sku': 3948, 'date': '2025.01.13', 'quantity': 48, 'warehouse': 'Astana'},
    {'sku': 2489, 'date': '2025.01.14', 'quantity': '13', 'warehouse': 'Kyzylorda'},
    {'sku': 5458, 'date': '2025.01.15', 'quantity': 10, 'warehouse': 'Almaty'}
]
rows[:3]

[{'sku': 1000, 'date': '2025.01.01', 'quantity': 10, 'warehouse': 'Almaty'},
 {'sku': 1050, 'date': '2025.01.02', 'quantity': 25, 'warehouse': 'Kostanay'},
 {'sku': 1809,
  'date': '2025.01.03',
  'quantity': 19,
  'warehouse': 'Kokshetau '}]

In [4]:
df_demo = pd.DataFrame(rows)

In [6]:
df_demo.to_csv('demo.csv', index=False, sep=';',
               encoding='utf-8')
print('saved: demo.csv')

saved: demo.csv


In [7]:
with pd.ExcelWriter('demo_exce.xlsx', engine='openpyxl') as writer:
  df_demo.to_excel(writer, sheet_name='data', index=False)
  pd.DataFrame({'note': ['лист для заметок'], 'value': [123]}).to_excel(writer, sheet_name='notes', index=False)
  print('saved: demo_exce.xlsx')

saved: demo_exce.xlsx


## **Импорт CSV: диагностика ошибок и корректное чтение**

In [8]:
df_bad = pd.read_csv('demo.csv')
df_bad.head()

,sku;date;quantity;warehouse
0,1000;2025.01.01;10;Almaty
1,1050;2025.01.02;25;Kostanay
2,1809;2025.01.03;19;Kokshetau
3,1309;2025.01.04;29;Astana
4,2348;2025.01.05;10;Almaty


In [9]:
print('shape:', df_bad.shape)
print(df_bad.columns)

shape: (15, 1)
Index(['sku;date;quantity;warehouse'], dtype='object')


In [10]:
df_csv = pd.read_csv('demo.csv',
                     sep=';',
                     encoding='utf-8',
                     na_values=['', 'NA', 'null']
                     )
df_csv.head()

,sku,date,quantity,warehouse
0,1000,2025.01.01,10,Almaty
1,1050,2025.01.02,25,Kostanay
2,1809,2025.01.03,19,Kokshetau
3,1309,2025.01.04,29,Astana
4,2348,2025.01.05,10,Almaty


In [11]:
df_csv['quantity'] = (df_csv['quantity']
                    .astype(str)
                    .str.replace(' ', '', regex=False)
                    .str.replace(',', '.', regex=False))
df_csv['quantity'] = pd.to_numeric(df_csv['quantity'], errors='coerce')

df_csv.head()

,sku,date,quantity,warehouse
0,1000,2025.01.01,10,Almaty
1,1050,2025.01.02,25,Kostanay
2,1809,2025.01.03,19,Kokshetau
3,1309,2025.01.04,29,Astana
4,2348,2025.01.05,10,Almaty


## **Импорт Excel: выбор листа и контроль структуры**

In [12]:
df_csv['warehouse'] = df_csv['warehouse'].str.strip()
df_csv.head()

,sku,date,quantity,warehouse
0,1000,2025.01.01,10,Almaty
1,1050,2025.01.02,25,Kostanay
2,1809,2025.01.03,19,Kokshetau
3,1309,2025.01.04,29,Astana
4,2348,2025.01.05,10,Almaty


In [13]:
xls = pd.ExcelFile('demo_exce.xlsx')
xls.sheet_names

['data', 'notes']

In [14]:
df_xlsx = pd.read_excel('demo_exce.xlsx', sheet_name='data')
df_xlsx.head()

,sku,date,quantity,warehouse
0,1000,2025.01.01,10,Almaty
1,1050,2025.01.02,25,Kostanay
2,1809,2025.01.03,19,Kokshetau
3,1309,2025.01.04,29,Astana
4,2348,2025.01.05,10,Almaty


## **Типизация столбцов и парсинг дат**

In [15]:
df_csv['date'] = pd.to_datetime(df_csv['date'], format='%Y.%m.%d',
                                errors='coerce')
df_csv[['date']].head()

,date
0,2025-01-01
1,2025-01-02
2,2025-01-03
3,2025-01-04
4,2025-01-05


In [16]:
df_csv['warehouse'] = df_csv['warehouse'].astype('category')
df_csv.dtypes

sku                   int64
date         datetime64[ns]
quantity              int64
warehouse          category
dtype: object

In [17]:
bad_dates = df_csv[df_csv['date'].isna()][['sku', 'date']]
bad_dates

,sku,date


## **Первичная валидация и диагностика ошибок импорта**

In [18]:
shape = df_csv.shape
dtypes = df_csv.dtypes.astype(str)
missing = df_csv.isna().sum()
missing_pct = (df_csv.isna().mean() * 100).round(1)
dup_full = int(df_csv.duplicated().sum())
dup_by_sku = int(df_csv.duplicated(subset=['sku']).sum())

In [19]:
print('shape: ', shape)
print('duplicates_full: ', dup_full)
print('duplicates_by_sku: ', dup_by_sku)
pd.DataFrame({'missing_count': missing, 'missing_percent': missing_pct})

shape:  (15, 4)
duplicates_full:  0
duplicates_by_sku:  3


,missing_count,missing_percent
sku,0,0.0
date,0,0.0
quantity,0,0.0
warehouse,0,0.0


## **Отчёт импорта**

In [20]:
import_report = pd.DataFrame({
    'Показатель': [
        'Файл', 'Формат', 'Параметры чтения', 'Число строк', 'Число столбцов', 'Дубликаты (полные)', 'Дубликаты (по sku)', 'Пропуски (всего)', 'Ошибки дат (NaT)'],
    'Значение': ['demo.csv', 'CSV', "sep=';'; encoding='utf-8'; na_values=['', 'NA', 'null']",
                 df_csv.shape[0], df_csv.shape[1], dup_full, dup_by_sku, int(df_csv.isna().sum().sum()), int(df_csv['date'].isna().sum())
    ]
})
import_report

,Показатель,Значение
0,Файл,demo.csv
1,Формат,CSV
2,Параметры чтения,"sep=';'; encoding='utf-8'; na_values=['', 'NA', 'null']"
3,Число строк,15
4,Число столбцов,4
5,Дубликаты (полные),0
6,Дубликаты (по sku),3
7,Пропуски (всего),0
8,Ошибки дат (NaT),0


## **КОНТРОЛЬНЫЕ ВОПРОСЫ**

**Какие признаки указывают на неверный разделитель при импорте CSV?**

Признаки неверного разделителя CSV:
* смещение столбцов
* лишние NaN
* объединение нескольких значений в одну ячейку

**Как кодировка влияет на корректность чтения текстовых столбцов?**

Неверная кодировка влияет на некорректное отображение символов (кракозябры), ошибки чтения текста.

**Почему важно явно типизировать столбцы и проверять dtypes после импорта?**

Зачем типизировать столбцы: это предотвращает ошибки анализа.
Проверка dtypes гарантирует корректные числовые и категориальные операции.

**Как диагностировать ошибки парсинга дат и какие стратегии их обработки возможны?**

При ошибках парсинга дат: отображается неверный формат, разные локали.
Стратегии - указать parse_dates, задать dayfirst, использовать pd.to_datetime с errors='coerce'.

**Какие параметры pandas.read_csv наиболее важны при реальных данных и почему?**

Важные параметры pandas.read_csv:
* sep
* encoding
* dtype
* parse_dates
* na_values
Определяют корректность структуры, текста, типов и пропусков.